## PHASE 2 — PART 3: LangChain Multi-Agent System
**Project:** Freshness Guard

4 Agents:
1. **Inventory Agent** — Stock, expiry, replenishment needs
2. **Demand Agent** — Sales forecasts and purchase patterns
3. **Weather Impact Agent** — Shelf-life reduction estimates
4. **Decision Agent** — Final freshness action recommendations

### Cell 1 — Imports & Ollama Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate

# Connect to Ollama — make sure 'ollama serve' is running in terminal
llm = OllamaLLM(model='llama3', temperature=0.3)

test = llm.invoke('Reply with only: Ollama is ready')
print(test)

### Cell 2 — Load All Data

In [ ]:
df = pd.read_csv('LangChain_Perfected_Freshness_Dataset.csv')
df['sales_date']    = pd.to_datetime(df['sales_date'])
df['expiry_date']   = pd.to_datetime(df['expiry_date'])
df['received_date'] = pd.to_datetime(df['received_date'])

forecast_df = pd.read_csv('demand_forecast_30days.csv')
forecast_df['date'] = pd.to_datetime(forecast_df['date'])

with open('demand_agent_context.txt', 'r', encoding='utf-8') as f:
    demand_context = f.read()

print('Dataset shape  :', df.shape)
print('Forecast shape :', forecast_df.shape)
print('Stores         :', df['store_id'].nunique())
print('All data loaded!')

### Cell 3 — Agent 1: Inventory Agent
**Answers:** Which store has highest spoilage risk? Which products are expiring soon?

In [ ]:
def run_inventory_agent(store_id, df, llm):
    store_df       = df[df['store_id'] == store_id].copy()
    total_stock    = store_df['stock_quantity'].sum()
    total_waste    = store_df['waste_quantity'].sum()
    waste_rate     = (total_waste / (total_stock + 1)) * 100
    danger_zone    = store_df[store_df['Days_to_Expiry_at_Sale'] <= 3]
    expiring_soon  = danger_zone.groupby('product')['stock_quantity'].sum().sort_values(ascending=False).head(5)
    top_waste      = store_df.groupby('product')['waste_quantity'].sum().sort_values(ascending=False).head(5)
    low_stock      = store_df[store_df['stock_quantity'] < 50].groupby('product')['stock_quantity'].mean().sort_values().head(5)

    context = (
        f'Store ID: {store_id}\n'
        f'Total Stock: {total_stock:,} units\n'
        f'Total Waste: {total_waste:,} units\n'
        f'Waste Rate: {waste_rate:.1f}%\n\n'
        f'Products expiring within 3 days:\n{expiring_soon.to_string()}\n\n'
        f'Top 5 most wasted products:\n{top_waste.to_string()}\n\n'
        f'Low stock products (under 50 units):\n{low_stock.to_string() if len(low_stock) > 0 else "None"}\n'
    )

    prompt = PromptTemplate(
        input_variables=['context'],
        template=(
            'You are an Inventory Agent for a grocery retail chain.\n'
            'Based on the inventory data below, give clear recommendations.\n'
            'Be concise. Use bullet points. Focus on actions for TODAY.\n\n'
            'Inventory Data:\n{context}\n\n'
            'Provide:\n'
            '1. Spoilage Risk Summary (1-2 sentences)\n'
            '2. Urgent Actions (bullet points)\n'
            '3. Replenishment Needs (bullet points)\n'
        )
    )
    response = (prompt | llm).invoke({'context': context})
    return response, context

print('=' * 60)
print('INVENTORY AGENT - Store_1')
print('=' * 60)
inv_response, inv_context = run_inventory_agent('Store_1', df, llm)
print(inv_response)

### Cell 4 — Agent 2: Demand Agent
**Answers:** Which store needs urgent replenishment? Is there a demand spike coming?

In [ ]:
def run_demand_agent(store_id, forecast_df, demand_context, llm):
    store_fc      = forecast_df[forecast_df['store_id'] == store_id].copy()
    total_30d     = store_fc['forecasted_demand'].sum()
    avg_daily     = store_fc['forecasted_demand'].mean()
    peak_day      = store_fc.loc[store_fc['forecasted_demand'].idxmax()]
    low_day       = store_fc.loc[store_fc['forecasted_demand'].idxmin()]
    store_fc['is_weekend'] = pd.to_datetime(store_fc['date']).dt.dayofweek >= 5
    weekend_avg   = store_fc[store_fc['is_weekend']]['forecasted_demand'].mean()
    weekday_avg   = store_fc[~store_fc['is_weekend']]['forecasted_demand'].mean()
    next_7        = store_fc.head(7)

    context = (
        f'Store ID: {store_id}\n'
        f'30-Day Total Forecasted Demand: {total_30d:,} units\n'
        f'Average Daily Demand: {avg_daily:.0f} units/day\n'
        f'Peak Demand Day: {peak_day["date"].strftime("%Y-%m-%d")} ({peak_day["forecasted_demand"]:,} units)\n'
        f'Lowest Demand Day: {low_day["date"].strftime("%Y-%m-%d")} ({low_day["forecasted_demand"]:,} units)\n'
        f'Weekend Average: {weekend_avg:.0f} units/day\n'
        f'Weekday Average: {weekday_avg:.0f} units/day\n\n'
        f'Next 7 Days Forecast:\n{next_7[["date","forecasted_demand"]].to_string(index=False)}\n\n'
        f'Overall Market Context:\n{demand_context}\n'
    )

    prompt = PromptTemplate(
        input_variables=['context'],
        template=(
            'You are a Demand Forecasting Agent for a grocery retail chain.\n'
            'Analyze the demand forecast and give replenishment recommendations.\n'
            'Be specific with quantities and dates. Use bullet points.\n\n'
            'Demand Forecast Data:\n{context}\n\n'
            'Provide:\n'
            '1. Demand Trend Summary (1-2 sentences)\n'
            '2. Replenishment Schedule (when and how much to order)\n'
            '3. Weekend Preparation Actions\n'
            '4. Risk Alerts (demand spikes or drops to watch)\n'
        )
    )
    response = (prompt | llm).invoke({'context': context})
    return response, context

print('=' * 60)
print('DEMAND AGENT - Store_1')
print('=' * 60)
dem_response, dem_context = run_demand_agent('Store_1', forecast_df, demand_context, llm)
print(dem_response)

### Cell 5 — Agent 3: Weather Impact Agent
**Answers:** Which products are at risk due to heat? How much does humidity reduce shelf life?

In [ ]:
def run_weather_agent(store_id, df, llm):
    store_df       = df[df['store_id'] == store_id].copy()
    avg_temp       = store_df['Temperature_C'].mean()
    max_temp       = store_df['Temperature_C'].max()
    avg_humidity   = store_df['Humidity_Percent'].mean()
    high_temp_days = (store_df['Temperature_C'] > 30).sum()
    weather_waste  = store_df.groupby('weather_severity_n')['waste_quantity'].mean().round(1)
    cat_impact     = store_df.groupby('category').agg(
        avg_waste      = ('waste_quantity', 'mean'),
        avg_temp       = ('Temperature_C',  'mean'),
        avg_shelf_life = ('shelf_life_days', 'mean')
    ).round(1).sort_values('avg_waste', ascending=False)
    heatwave_risk  = 'HIGH' if avg_temp > 28 else 'MEDIUM' if avg_temp > 24 else 'LOW'

    context = (
        f'Store ID: {store_id}\n'
        f'Average Temperature: {avg_temp:.1f}C\n'
        f'Maximum Temperature: {max_temp:.1f}C\n'
        f'Average Humidity: {avg_humidity:.1f}%\n'
        f'High Temp Days (above 30C): {high_temp_days}\n'
        f'Heatwave Risk Level: {heatwave_risk}\n\n'
        f'Waste by weather severity (0=mild,1=moderate,2=severe):\n{weather_waste.to_string()}\n\n'
        f'Category temperature and waste impact:\n{cat_impact.to_string()}\n'
    )

    prompt = PromptTemplate(
        input_variables=['context'],
        template=(
            'You are a Weather Impact Agent for a grocery retail chain.\n'
            'Analyze weather data and estimate its impact on product shelf life and spoilage.\n'
            'Give specific actionable advice. Use bullet points.\n\n'
            'Weather and Spoilage Data:\n{context}\n\n'
            'Provide:\n'
            '1. Weather Risk Summary (1-2 sentences)\n'
            '2. High Risk Product Categories\n'
            '3. Shelf Life Reduction Estimates\n'
            '4. Recommended Storage and Discount Actions\n'
        )
    )
    response = (prompt | llm).invoke({'context': context})
    return response, context

print('=' * 60)
print('WEATHER IMPACT AGENT - Store_1')
print('=' * 60)
weather_response, weather_context = run_weather_agent('Store_1', df, llm)
print(weather_response)

### Cell 6 — Agent 4: Decision Agent (Master Orchestrator)
**Role:** Combines all 3 agent outputs and generates final freshness action recommendations like:
- *Heatwave predicted -> Reduce leafy greens by 15%*
- *Demand spike -> Increase milk replenishment in Store_1*
- *Slow-moving stock -> Apply 20% discount today*

In [ ]:
def run_decision_agent(store_id, inv_response, dem_response, weather_response, llm):
    context = (
        f'STORE: {store_id}\n\n'
        f'--- INVENTORY AGENT REPORT ---\n{inv_response}\n\n'
        f'--- DEMAND AGENT REPORT ---\n{dem_response}\n\n'
        f'--- WEATHER IMPACT AGENT REPORT ---\n{weather_response}\n'
    )

    prompt = PromptTemplate(
        input_variables=['context', 'store_id'],
        template=(
            'You are the Decision Agent for a grocery retail freshness optimization system.\n'
            'You received reports from 3 specialist agents: Inventory, Demand, and Weather Impact.\n'
            'Synthesize all insights and produce final freshness action recommendations.\n\n'
            'Use this EXACT format for each recommendation:\n'
            '[ACTION TYPE] Reason -> Specific Action\n\n'
            'Examples:\n'
            '[DISCOUNT] Slow-moving dairy nearing expiry -> Apply 25% discount on milk today\n'
            '[REPLENISHMENT] Demand spike this weekend -> Order 200 extra units of bread by Friday\n'
            '[REDISTRIBUTE] Excess stock + low demand -> Transfer 50 units of yogurt to Store_4\n'
            '[REDUCE ORDER] Heatwave predicted -> Reduce leafy greens order by 15% this week\n\n'
            'Agent Reports:\n{context}\n\n'
            'Generate 5-7 specific freshness recommendations for {store_id}.\n'
            'Each must follow the [ACTION TYPE] format.\n'
            'Be specific with product names, quantities, percentages, and dates.\n'
        )
    )
    response = (prompt | llm).invoke({'context': context, 'store_id': store_id})
    return response

print('=' * 60)
print('DECISION AGENT - Store_1 FINAL RECOMMENDATIONS')
print('=' * 60)
decision_response = run_decision_agent(
    'Store_1', inv_response, dem_response, weather_response, llm
)
print(decision_response)

### Cell 7 — Full Pipeline: All Agents for All Stores
Runs the complete 4-agent pipeline for every store and saves all recommendations.

In [ ]:
stores = sorted(df['store_id'].unique())
all_recommendations = {}

print('Running full LangChain agent pipeline for all stores...')
print('This may take several minutes as Ollama processes each store.\n')

for store in stores:
    print(f'Processing {store}...')
    try:
        inv_r,     _ = run_inventory_agent(store, df, llm)
        dem_r,     _ = run_demand_agent(store, forecast_df, demand_context, llm)
        weather_r, _ = run_weather_agent(store, df, llm)
        decision_r   = run_decision_agent(store, inv_r, dem_r, weather_r, llm)
        all_recommendations[store] = {
            'inventory' : inv_r,
            'demand'    : dem_r,
            'weather'   : weather_r,
            'decision'  : decision_r
        }
        print(f'  Done {store}')
    except Exception as e:
        print(f'  Error for {store}: {e}')
        all_recommendations[store] = {'error': str(e)}

# Save all recommendations
with open('freshness_recommendations.txt', 'w', encoding='utf-8') as f:
    for store, recs in all_recommendations.items():
        f.write('\n' + '='*60 + '\n')
        f.write(f'STORE: {store}\n')
        f.write('='*60 + '\n')
        if 'error' in recs:
            f.write(f'Error: {recs["error"]}\n')
        else:
            f.write('\n--- DECISION AGENT RECOMMENDATIONS ---\n')
            f.write(recs['decision'] + '\n')

print('\nAll stores processed!')
print('Saved: freshness_recommendations.txt')

### Cell 8 — Final Recommendations Dashboard
Clean summary of all AI recommendations across all stores.

In [ ]:
print('FRESHNESS GUARD - AI RECOMMENDATIONS DASHBOARD')
print('=' * 60)

for store, recs in all_recommendations.items():
    print(f'\nSTORE: {store}')
    print('-' * 40)
    if 'error' in recs:
        print(f'  Could not process: {recs["error"]}')
    else:
        lines = recs['decision'].split('\n')
        for line in lines:
            if line.strip().startswith('['):
                print(f'  {line.strip()}')

print('\nPhase 2 LangChain Agents COMPLETE!')
print('Next Step: Phase 3 - PuLP Optimization + Interactive Dashboard')